# <span style="color:#0097b2">0. Data</span>
[Back to TOC](#toc)

## <h3 style="color: #333333; background-color: #b0cece; text-align: center; padding: 6px; font-family: 'Arial';">Libraries Importation</h3>

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import AgglomerativeClustering
from scipy.cluster.hierarchy import dendrogram, linkage

## <h3 style="color: #333333; background-color: #b0cece; text-align: center; padding: 6px; font-family: 'Arial';">Data Importation</h3>

In [2]:
df_eng = pd.read_csv('../data/engagement_based_perspective_clusters.csv', index_col='Loyalty#')
df_beh = pd.read_csv('../data/behvioral_segment_data.csv', index_col='Loyalty#')
df_val = pd.read_csv('../data/value_based_perspective_clusters.csv', index_col='Loyalty#')

# <span style="color:#0097b2">1. Data Preparation & Label Extraction</span>
[Back to TOC](#toc)

In [3]:
df_merged = df_val.copy()

# Add missing columns from Behavior DF
cols_to_add_beh = df_beh.columns.difference(df_merged.columns)
df_merged = df_merged.join(df_beh[cols_to_add_beh])

# Add missing columns from Engagement DF
cols_to_add_eng = df_eng.columns.difference(df_merged.columns)
df_merged = df_merged.join(df_eng[cols_to_add_eng])

In [4]:
df_merged

,Income,Customer Lifetime Value,EnrollmentType,rejoined_program,Days_in_prog,Cancelled_program,Enrollment_year,Enrollment_month,Location_cluster,TotalFlights,...,seg_Fragile_Value_Contributers,seg_Premium_driven_value,seg_Active_Redeemers,seg_From_September_to_December_travelers,seg_Holiday_travelers,seg_No_Flights,seg_Active_Travelers,seg_Dormant_Travelers,seg_Low_Engagement,seg_Non_Travelers
Loyalty#,,,,,,,,,,,,,,,,,,,,,
100011,34148.0,5780.18,0,0,0.0,0,2017,5,1,0.0,...,1,0,0,0,0,1,0,0,0,1
100012,34148.0,5780.18,0,0,0.0,0,2019,2,4,0.0,...,1,0,0,0,0,1,0,0,0,1
100013,34148.0,5780.18,0,0,0.0,0,2017,9,5,0.0,...,1,0,0,0,0,1,0,0,0,1
100014,34148.0,5780.18,0,0,0.0,0,2020,11,3,0.0,...,1,0,0,0,0,1,0,0,0,1
100015,34148.0,5780.18,0,0,0.0,0,2020,4,4,0.0,...,1,0,0,0,0,1,0,0,0,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
999995,34148.0,5780.18,0,0,0.0,0,2020,3,4,0.0,...,1,0,0,0,0,1,0,0,0,1
999996,34148.0,5780.18,0,0,0.0,0,2018,9,1,0.0,...,1,0,0,0,0,1,0,0,0,1
999997,34148.0,5780.18,0,0,0.0,0,2020,1,3,0.0,...,1,0,0,0,0,1,0,0,0,1


In [5]:
def decode_ohe_segments(df, prefix):
    seg_cols = [c for c in df.columns if c.startswith(prefix)]
    return df[seg_cols].idxmax(axis=1)


In [6]:
df_merged['Value_Label'] = decode_ohe_segments(
    df_val, prefix='seg_'
)

df_merged['Behavior_Label'] = decode_ohe_segments(
    df_beh, prefix='seg_'
)

df_merged['Engagement_Label'] = decode_ohe_segments(
    df_eng, prefix='seg_'
)

In [7]:
df_merged['Value_Label_Code'] = pd.factorize(df_merged['Value_Label'])[0]
df_merged['Behavior_Label_Code'] = pd.factorize(df_merged['Behavior_Label'])[0]
df_merged['Engagement_Label_Code'] = pd.factorize(df_merged['Engagement_Label'])[0]

In [11]:
features_to_exclude = ['Value_Label_Code', 'Behavior_Label_Code', 'Engagement_Label_Code']

# Also exclude any One-Hot Encoded segment columns (starting with 'seg_')
# ensuring we don't count the labels twice
features_to_exclude += [c for c in df_merged.columns if c.startswith('seg_')]
features_to_exclude += [c for c in df_merged.columns if c.startswith('cluster_')]

# but this logic works if your dataframe only contains relevant metric columns.
metric_features = (
    df_merged
    .drop(columns=features_to_exclude)
    .select_dtypes(include='number')
    .columns
    .tolist()
)


In [12]:
df_merged.groupby(['Value_Label', 'Behavior_Label', 'Engagement_Label'])[metric_features].mean()

Income  \
Value_Label                    Behavior_Label                           Engagement_Label                      
seg_Behavior_sustained_value   seg_Active_Redeemers                     seg_Active_Travelers   34823.718421   
                                                                        seg_Dormant_Travelers  20976.000000   
                                                                        seg_Low_Engagement     35149.096774   
                               seg_From_September_to_December_travelers seg_Active_Travelers   34599.714286   
                                                                        seg_Dormant_Travelers  33951.218750   
                                                                        seg_Low_Engagement     35919.702312   
                               seg_Holiday_travelers                    seg_Active_Travelers   34798.793536   
                                                                        seg_Low_Engagement     36082.091954   
seg_Efficiency_Optimized_Value seg_Active_Redeemers                     seg_Active_Travelers    1975.900079   
                                                                        seg_Low_Engagement      1525.939189   
                               seg_From_September_to_December_travelers seg_Active_Travelers    1501.742857   
                                                                        seg_Dormant_Travelers    678.396825   
                                                                        seg_Low_Engagement      1637.236842   
                               seg_Holiday_travelers                    seg_Active_Travelers    2311.544183   
                                                                        seg_Low_Engagement      2339.016393   
seg_Fragile_Value_Contributers seg_Active_Redeemers                     seg_Active_Travelers   21688.615385   
                                                                        seg_Low_Engagement     18964.219512   
                               seg_From_September_to_December_travelers seg_Active_Travelers   15546.750000   
                                                                        seg_Dormant_Travelers  31613.672680   
                                                                        seg_Low_Engagement     34417.985133   
                               seg_Holiday_travelers                    seg_Active_Travelers   17143.583333   
                                                                        seg_Low_Engagement     14676.000000   
                               seg_No_Flights                           seg_Non_Travelers      38324.439210   
seg_Premium_driven_value       seg_Active_Redeemers                     seg_Active_Travelers   74692.665460   
                                                                        seg_Dormant_Travelers  74015.000000   
                                                                        seg_Low_Engagement     70892.435897   
                               seg_From_September_to_December_travelers seg_Active_Travelers   74812.112500   
                                                                        seg_Dormant_Travelers  60501.748571   
                                                                        seg_Low_Engagement     58731.175538   
                               seg_Holiday_travelers                    seg_Active_Travelers   74929.657343   
                                                                        seg_Low_Engagement     74810.893939   
                               seg_No_Flights                           seg_Non_Travelers      35377.636816   

                                                                                               Customer Lifetime Value  \
Value_Label                    Behavior_Label                           Engagement_Label                                 
seg_Behavior_sustained_value   seg_Active_Redeemers                     seg_Active

# <span style="color:#0097b2">2. Micro-Segmentation Strategy</span>
[Back to TOC](#toc)

In [ ]:
df_merged['Micro_Segment'] = (
    df_merged['Value_Label'].astype(str) + " | " + 
    df_merged['Behavior_Label'].astype(str) + " | " + 
    df_merged['Engagement_Label'].astype(str)
)

print(f"Number of Micro-Segments formed: {df_merged['Micro_Segment'].nunique()}")

In [ ]:
df_merged

In [ ]:
crosstab = pd.crosstab(df_merged['Engagement_Label'], df_merged['Engagement_Label'])
crosstab

In [ ]:
features_to_exclude = ['Micro_Segment', 'Value_Label', 'Behavior_Label', 'Engagement_Label']

# Also exclude any One-Hot Encoded segment columns (starting with 'seg_')
# ensuring we don't count the labels twice
features_to_exclude += [c for c in df_merged.columns if c.startswith('seg_')]
features_to_exclude += [c for c in df_merged.columns if c.startswith('cluster_')]

# 3. Define Metric Features (The variables to calculate Centroids on)
# NOTE: As discussed, it is safer to use an explicit list of "Active Variables",
# but this logic works if your dataframe only contains relevant metric columns.
metric_features = [c for c in df_merged.columns if c not in features_to_exclude]

In [ ]:
# Cell 16: Calculate centroids for each combination
df_centroids = df_merged.groupby(['Value_Label', 'Behavior_Label', 'Engagement_Label'])[metric_features].mean()
df_centroids

In [ ]:
features_to_exclude = ['Micro_Segment', 'Value_Label', 'Behavior_Label', 'Engagement_Label']

# Also exclude any One-Hot Encoded segment columns (starting with 'seg_')
# ensuring we don't count the labels twice
features_to_exclude += [c for c in df_merged.columns if c.startswith('seg_')]
features_to_exclude += [c for c in df_merged.columns if c.startswith('cluster_')]

# 3. Define Metric Features (The variables to calculate Centroids on)
# NOTE: As discussed, it is safer to use an explicit list of "Active Variables",
# but this logic works if your dataframe only contains relevant metric columns.
metric_features = [c for c in df_merged.columns if c not in features_to_exclude]

print(f"Calculating centroids based on {len(metric_features)} features.")

# 4. Calculate Centroids (Mean of features per micro-segment)
df_centroids = df_merged.groupby('Micro_Segment')[metric_features].mean()

# 5. Scale the centroids before Hierarchical Clustering
scaler = StandardScaler()
centroids_scaled = scaler.fit_transform(df_centroids)

df_centroids_scaled = pd.DataFrame(
    centroids_scaled, 
    index=df_centroids.index, 
    columns=df_centroids.columns
)

# Check the result
print(df_centroids_scaled.head())

# <span style="color:#0097b2">3. Hierarchical Clustering (The Merge)</span>
[Back to TOC](#toc)

In [ ]:
plt.figure(figsize=(12, 6))
plt.title('Dendrogram of Micro-Segment Centroids')
dendrogram = dendrogram(linkage(df_centroids_scaled, method='ward'))
plt.xlabel('Micro-Segments')
plt.ylabel('Euclidean Distance')
plt.show()

In [ ]:
n_final_clusters = 8

hclust_final = AgglomerativeClustering(
    linkage='ward',
    n_clusters=n_final_clusters
)

hclust_final_labels = hclust_final.fit_predict(df_centroids)
hclust_final_labels

In [ ]:
# Add to centroids dataframe
df_centroids['merged_labels'] = hclust_final_labels
df_centroids

In [ ]:
hc = AgglomerativeClustering(n_clusters=8, linkage='ward')
df_centroids['Final_Cluster_Label'] = hc.fit_predict(df_centroids_scaled)

In [ ]:
# Create a mapping dictionary: Micro-Segment -> Final Cluster
micro_to_final_map = df_centroids['Final_Cluster_Label'].to_dict()

# Map the final cluster back to the main customer dataframe
df_merged['Final_Solution_Label'] = df_merged['Micro_Segment'].map(micro_to_final_map)

In [ ]:
print("Final Segment Distribution:")
print(df_merged['Final_Solution_Label'].value_counts().sort_index())

# Display a sample of how perspectives were merged
print("\nSample mapping of Perspectives to Final Solution:")
print(df_merged[['Engagement_Label', 'Behavior_Label', 'Value_Label', 'Micro_Segment', 'Final_Solution_Label']].head(10))

In [ ]:
def get_rsq(df, feats, label_col):
    """
    Calculate the R-squared value for a given DataFrame and features.
    
    Parameters:
    df (pd.DataFrame): The input DataFrame containing the data.
    feats (list): A list of feature column names to be used in the calculation.
    label_col (str): The name of the column containing the labels or cluster assignments.
    
    Returns:
    float: The R-squared value, representing the proportion of variance explained by the clustering.
    """
    df_sst_ = get_ss(df, feats)  # get total sum of squares
    df_ssw_ = get_ssw(df, feats, label_col)  # get ss within
    df_ssb_ = df_sst_ - df_ssw_  # get ss between
    # r2 = ssb/sst
    return (df_ssb_ / df_sst_)

In [ ]:
# Quick distinctiveness check (R²)
r2_merged = get_rsq(df_merged, metric_features, 'merged_labels')
print(f"\nMerged clustering R²: {r2_merged:.4f}")

# <span style="color:#0097b2">4. Profiling & Feature Importance</span>
[Back to TOC](#toc)

In [ ]:
# 1. Calculate R2 for ALL variables (Active + Passive)
# (excluding ID columns or text columns)
numeric_cols = df_merged.select_dtypes(include=[np.number]).columns.tolist()
# Remove the cluster labels themselves
numeric_cols = [c for c in numeric_cols if c not in ['Final_Solution_Label', 'Location_cluster', 'EnrollmentType']]

def get_r2_scores(df, cluster_col, features):
    sst_total = df[features].var() * (len(df) - 1)
    r2_scores = {}
    for feature in features:
        sst = sst_total[feature]
        if sst == 0: continue # Skip constant columns
        ssw = 0
        for cluster in df[cluster_col].unique():
            cluster_data = df[df[cluster_col] == cluster][feature]
            ssw += cluster_data.var() * (len(cluster_data) - 1)
        r2_scores[feature] = (sst - ssw) / sst
    return pd.Series(r2_scores).sort_values(ascending=False)

In [ ]:
# 2. Get Top 20 Differentiating Features
top_features = get_r2_scores(df_merged, 'Final_Solution_Label', numeric_cols).head(20).index.tolist()

# 3. Plot Heatmap ONLY for these Top 20
cluster_means = df_merged.groupby('Final_Solution_Label')[top_features].mean()
global_means = df_merged[top_features].mean()
relative_imp = (cluster_means / global_means) - 1

plt.figure(figsize=(10, 12))
sns.heatmap(data=relative_imp.T, annot=True, fmt='.2f', cmap='RdBu_r', center=0)
plt.title('Top 20 Most Distinctive Features')
plt.show()

In [ ]:
# Create a temporary dataframe for plotting
seasonality_cols = [f'Pct_Spend_Month_{i}' for i in range(1, 13)]

# Group by Cluster and get the mean of monthly spend
monthly_profile = df_merged.groupby('Final_Solution_Label')[seasonality_cols].mean()

# Rename columns to just numbers 1-12 for the X-axis
monthly_profile.columns = range(1, 13)

# Plot
plt.figure(figsize=(12, 6))
# Transpose so Months are on X-axis
sns.lineplot(data=monthly_profile.T, dashes=False, palette='tab10', linewidth=2.5)

plt.title('Spending Seasonality by Segment')
plt.xlabel('Month')
plt.ylabel('Percent of Spend')
plt.xticks(range(1, 13))
plt.legend(title='Cluster')
plt.grid(True, alpha=0.3)
plt.show()